# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nimrafarooq2006-coder/ml-internship-work/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Two findings from the FlyRank paper (State of AI-Driven SEO, March
2026), and the methodology question I would ask about each. I ask
them to practise the same careful reading on my own work, not to
grade the paper. The paper is clear that it is observational and
exploratory, which I appreciate.

FINDING 1: "The Anatomy of Growing Content". Growing pages are longer
(3.2K vs 2.3K words) and younger (184 vs 230 days) than declining
pages.

My question: where does the "growing" label come from, and could it
be partly built into the comparison? The paper defines trend
direction from the 30-day versus previous-30-day change in
impressions (up above +10%, down below -10%). Pages that are young
or recently updated may be moving up from a low starting point, so
the age gap could reflect that ramp-up and not a lasting pattern. A
second question: the comparison covers 57 brands together. If brands
with longer, younger content also happen to grow more, the gap could
come from the brand mix. Would the gap hold inside each brand, and
would medians tell the same story as averages for skewed measures
like word count? The paper already calls this directional and
observational, so these checks would show how robust it is.

FINDING 2: the exploratory growth model in the ML appendix. A
logistic regression reaches "71% holdout accuracy" on an 80/20 split.

My question: does the validation design support that number for new
brands? The paper does not say whether the 80/20 split is random or
grouped by brand. In my own data, a random split put 32 clients in
both train and test, and the random forest's Precision@50 fell from
0.98 to 0.60 once whole clients were held out. A second question:
what accuracy would a model that always says "growing" get? In
Finding 1 there are about 74.8K rising and 45.6K falling pages, so if
the classes are similarly imbalanced, always saying "growing" would
already score about 62%, and 71% would be a smaller gain than it
first looks. The paper presents the model as descriptive, so this
would mainly show how far the 71% generalises.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

Before and after. I trained the same two models under two splits.
BEFORE: a random split of pages, where 32 clients appear in both
train and test. AFTER: whole clients held out, with 0 clients in
both.

Random forest: Precision@50 fell from 0.98 to 0.60, Precision@100
from 0.91 to 0.55, and AUC from 0.745 to 0.597. Logistic regression
changed much less (Precision@50 0.66 to 0.66, AUC 0.606 to 0.576).

This suggests the random split flattered the more flexible model:
pages from the same client look alike, so a forest can pick up
client-specific patterns that do not carry over to new clients. The
honest split is the one that matches how the model would be used, on
clients it has not seen. All the numbers in my other notebooks use the
honest split. These are measured results on a 30,000-page starter
slice with 10 test clients, so they are directional.

In [4]:
import os, subprocess
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(path):
    if not os.path.isdir("ml-internship-work"):
        subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/nimrafarooq2006-coder/ml-internship-work"], check=True)
    path = "ml-internship-work/" + path

df = pd.read_csv(path)
df["declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["declining"].values
groups = df["client_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def make_models():
    return {
        "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "Random forest": RandomForestClassifier(n_estimators=200, max_depth=8,
                            min_samples_leaf=50, n_jobs=-1, random_state=42),
    }

idx = np.arange(len(df))
splits = {
    "BEFORE: random split": train_test_split(idx, test_size=0.3, random_state=42),
    "AFTER: whole clients held out": next(GroupShuffleSplit(test_size=0.3, random_state=42).split(X, y, groups)),
}

rows = []
for split_name, (tr, te) in splits.items():
    overlap = len(set(groups.iloc[tr]) & set(groups.iloc[te]))
    for name, m in make_models().items():
        m.fit(X.iloc[tr], y[tr])
        s = m.predict_proba(X.iloc[te])[:, 1]
        rows.append({"Split": split_name, "Clients in both train and test": overlap, "Model": name,
                     "Precision@50": precision_at_k(s, y[te], 50),
                     "Precision@100": precision_at_k(s, y[te], 100),
                     "AUC": roc_auc_score(y[te], s)})

pd.DataFrame(rows).round(3)

,Split,Clients in both train and test,Model,Precision@50,Precision@100,AUC
0,BEFORE: random split,32,Logistic regression,0.66,0.62,0.606
1,BEFORE: random split,32,Random forest,0.98,0.91,0.745
2,AFTER: whole clients held out,0,Logistic regression,0.66,0.73,0.576
3,AFTER: whole clients held out,0,Random forest,0.60,0.55,0.597


## 3. Leakage audit

Leakage audit. 1) No single feature is suspiciously predictive: the
best, word_count, reaches a single-feature AUC of only 0.563 on the
held-out clients (a leaked column would sit near 1.0). 2) The data
holds two label-related columns, trend_direction and trend_pct. The
label is trend_direction == "down", and trend_pct is the percentage
change it is computed from, so neither is used as a feature (the
features are content_age_days, days_since_last_update,
impressions_90d, avg_position, ctr and word_count). 3) There are no
duplicate pages (30,000 rows, 30,000 distinct content_id). 4) No page
appears in both train and test, and no client does either.

What this does not rule out: the label itself is a simple proxy
calculated from the same window as the features, so it is not a true
future outcome. A page's measured trend and its features describe the
same period, so the model shows association, not prediction of what
will happen later.

In [5]:
tr, te = splits["AFTER: whole clients held out"]

print("1. Is any single feature suspiciously good at predicting the label?")
rows = []
for f in features:
    a = roc_auc_score(y[te], X.iloc[te][f])
    rows.append({"feature": f, "single_feature_AUC": round(max(a, 1 - a), 3)})
print(pd.DataFrame(rows).sort_values("single_feature_AUC", ascending=False).to_string(index=False))

print("\n2. Columns related to the label that could hide among my features:")
print("   Label-related columns in the data:", [c for c in df.columns if "trend" in c.lower()])
print("   Used as features:", [c for c in features if "trend" in c.lower()])

print("\n3. Duplicate pages?")
print("   Rows:", len(df), "| distinct content_id:", df["content_id"].nunique())

print("\n4. Same page in both train and test:",
      len(set(df["content_id"].iloc[tr]) & set(df["content_id"].iloc[te])))


1. Is any single feature suspiciously good at predicting the label?
               feature  single_feature_AUC
            word_count               0.563
      content_age_days               0.547
                   ctr               0.536
days_since_last_update               0.514
       impressions_90d               0.513
          avg_position               0.501

2. Columns related to the label that could hide among my features:
   Label-related columns in the data: ['trend_direction', 'trend_pct']
   Used as features: []

3. Duplicate pages?
   Rows: 30000 | distinct content_id: 30000

4. Same page in both train and test: 0


## 4. Claim rewrite

Claims I made earlier, and how I would state them now:

1. Before: "Stale pages decline more." After: Pages 91-180 days since
   their last update were marked declining more often (61.1%) than
   pages updated in the last 90 days (51.2%) in the starter slice. The
   groups over 180 days are too small to judge. This is an observed
   association.

2. Before: "My rule picks pages that are declining." After: On all
   30,000 pages the rule's flagged group declined more often (62.3%
   against 51.2%), but on held-out clients its top 50 was only 46%
   declining, below the 55.9% that random picking gives. It is a weak
   baseline.

3. Before: "Logistic regression is the best model." After: Logistic
   regression had the best top-50 and top-100 precision on the 10 held-
   out clients (0.66 and 0.73), but overall AUC is similar across
   models (0.57 to 0.60). With only 10 test clients the differences are
   directional.

4. Before: "Content age is the most important feature." After: In this
   model, shuffling content age reduced AUC the most. This shows what
   the model uses, not what causes a decline, and age is related to
   days since the last update.

5. Before: "The model finds pages to refresh." After: The model offers
   a rough review order for decision support. It does not show that a
   refresh will help a page.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.